In [1]:
import pandas as pd

In [2]:
train_transaction_df = pd.read_csv(r'D:\Projects\Intellectra\ML-KOMPETISI-INTELLECTRA-2025-IPB\dataset\train_transaction_data.csv', low_memory=False)
train_label_df = pd.read_csv(r'D:\Projects\Intellectra\ML-KOMPETISI-INTELLECTRA-2025-IPB\dataset\train_label_data.csv', low_memory=False)

In [3]:
if train_transaction_df['PricePerUnit'].isnull().sum() > 0:
    median_price_per_unit = train_transaction_df['PricePerUnit'].median()
    #train_transaction_df['PricePerUnit'].fillna(median_price_per_unit, inplace=True)
    train_transaction_df['PricePerUnit'] = train_transaction_df['PricePerUnit'].fillna(median_price_per_unit)
    print(f"Filled missing PricePerUnit with median: {median_price_per_unit:.2f}")
else:
    print("No missing PricePerUnit found.")

train_transaction_df['TotalPrice'] = train_transaction_df['Qty'] * train_transaction_df['PricePerUnit']

Filled missing PricePerUnit with median: 265000.00


In [4]:
train_transaction_df['TransactionDatetime'] = pd.to_datetime(train_transaction_df['TransactionDatetime'])
snapshot_date = train_transaction_df['TransactionDatetime'].max() + pd.Timedelta(days=1)
print(f"\nUsing snapshot_date for recency calculation: {snapshot_date}")


Using snapshot_date for recency calculation: 2021-07-01 16:02:00+00:00


In [5]:
rfm_features_df = train_transaction_df.groupby('MemberID').agg(
    # Recency: Days since last purchase
    last_purchase_date=('TransactionDatetime', 'max'),
    # Frequency: Number of unique transactions
    frequency=('TransactionID', 'nunique'),
    # Monetary: Sum of TotalPrice
    monetary=('TotalPrice', 'sum')
).reset_index()

rfm_features_df['recency'] = (snapshot_date - rfm_features_df['last_purchase_date']).dt.days
rfm_features_df.drop(columns=['last_purchase_date'], inplace=True)

In [6]:
final_train_df = pd.merge(train_label_df, rfm_features_df, on='MemberID', how='left')
max_recency_val = final_train_df['recency'].max() if not final_train_df['recency'].isnull().all() else 9999
    
final_train_df['recency'].fillna(max_recency_val + 1, inplace=True) # Set to a value indicating very old/no purchase
final_train_df['frequency'].fillna(0, inplace=True)
final_train_df['monetary'].fillna(0, inplace=True)


print("\n--- RFM Features DataFrame Head (after merging with labels) ---")
print(final_train_df.head())

print("\n--- RFM Features DataFrame Info (after merging with labels) ---")
print(final_train_df.info())


--- RFM Features DataFrame Head (after merging with labels) ---
                           MemberID  next_buy  frequency   monetary  recency
0  7ef72aa51aecb701dc5c4074480fcdf6         0         10  1689000.0       66
1  577f1b9a093c2cec6398b1118f5d99ab         0         10  2478000.0      147
2  e2ee74f248a74ed886a22f14348fbafd         0          1   263500.0      365
3  cefa8ef7469a8b4e6df3f745d4905000         0          2   592800.0      334
4  5eecda17ddf06ed9d79f298b13f84785         0         11  2122200.0      311

--- RFM Features DataFrame Info (after merging with labels) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40020 entries, 0 to 40019
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   MemberID   40020 non-null  object 
 1   next_buy   40020 non-null  int64  
 2   frequency  40020 non-null  int64  
 3   monetary   40020 non-null  float64
 4   recency    40020 non-null  int64  
dtypes: float64(1), 

C:\Users\Irvan\AppData\Local\Temp\ipykernel_29692\40817157.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_train_df['recency'].fillna(max_recency_val + 1, inplace=True) # Set to a value indicating very old/no purchase
C:\Users\Irvan\AppData\Local\Temp\ipykernel_29692\40817157.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object o